In [127]:
import requests
import pandas as pd
import sqlite3

In [128]:
url = "https://px.hagstofa.is:443/pxen/api/v1/en/Efnahagur/visitolur/1_vnv/1_vnv/VIS01000.px"


query = {
  "query": [
    {
      "code": "Index",
      "selection": {
        "filter": "item",
        "values": [
          "CPI"
        ]
      }
    },
    {
      "code": "Item",
      "selection": {
        "filter": "item",
        "values": [
          "index"
        ]
      }
    }
  ],
  "response": {
    "format": "json"
  }
}

In [129]:
r = requests.post(url, json=query)
print(r.status_code)


200


In [130]:
df = pd.DataFrame([
    {
        "Date": pd.to_datetime(d["key"][0], format="%YM%m").strftime("%Y-%m-%d"),
        "CPI": d["values"][0]
    }
    for d in r.json()["data"]
])

df = df.sort_values("Date")
df["CPI"] = pd.to_numeric(df["CPI"], errors="coerce")
df["Updated"] = pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds")
df["CPI_annual_pct_change"] = round(df["CPI"].pct_change(12) * 100,3)

In [131]:
df

,Date,CPI,Updated,CPI_annual_pct_change
0,1988-05-01,100.0,2026-11-27T13:50:13+00:00,NaN
1,1988-06-01,103.4,2026-11-27T13:50:13+00:00,NaN
2,1988-07-01,107.0,2026-11-27T13:50:13+00:00,NaN
3,1988-08-01,109.3,2026-11-27T13:50:13+00:00,NaN
4,1988-09-01,110.0,2026-11-27T13:50:13+00:00,NaN
...,...,...,...,...
458,2026-07-01,693.2,2026-11-27T13:50:13+00:00,5.254
459,2026-08-01,694.6,2026-11-27T13:50:13+00:00,5.627
460,2026-09-01,697.3,2026-11-27T13:50:13+00:00,5.924
461,2026-10-01,700.4,2026-11-27T13:50:13+00:00,5.897


In [132]:
with sqlite3.connect("inflation.db") as conn:
    existing = pd.read_sql("SELECT * FROM cpi_base", conn)

    conn.executemany("""
        INSERT INTO cpi_base (Date, CPI, CPI_annual_pct_change, Updated)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(Date) DO UPDATE SET
            CPI = excluded.CPI,
            CPI_annual_pct_change = excluded.CPI_annual_pct_change,
            Updated = excluded.Updated
        WHERE (
            excluded.CPI IS NOT cpi_base.CPI
            OR excluded.CPI_annual_pct_change IS NOT cpi_base.CPI_annual_pct_change
        )
        AND excluded.Updated > cpi_base.Updated
    """, df[["Date", "CPI", "CPI_annual_pct_change", "Updated"]]
        .astype(object)
        .where(pd.notna(df), None)
        .itertuples(index=False, name=None))